# A.R.I.D. YOLOv5s v2: fine-tune for new environments

Improves the current model (`best.pt`, test mAP@50 0.904) so it fires less on clean scenes and handles more environments. It stays on **YOLOv5s, 5 classes, 640 × 640, ONNX opset 12**, so the web dashboard needs no code changes.

What this notebook changes compared with v1:

1. **Starts from your `best.pt`** instead of generic `yolov5s.pt`.
2. **Adds non-breeding scenes** (images with no boxes): your own photos plus a filtered sample of Places365 outdoor and indoor scenes. v1 had only 38 of 3,871 such images, which is why it detects objects everywhere.
3. **Optionally merges extra Roboflow datasets**, renaming their classes to the five A.R.I.D. classes.
4. **Keeps the original test split untouched**, so old and new scores are directly comparable. It also measures the false-alarm rate on clean scenes held out **before** any screening, so the comparison is fair.
5. **Learns from its mistakes:** photos in `hard_negatives/` (clean scenes the model wrongly flagged, checked by you) are added to training.
6. **Trains on the Colab disk and copies results to Drive afterwards.** In v1, a Drive disconnect crashed training.

**Before running**, put these in `MyDrive/ARID-YOLO/` in Google Drive:

| Item | Required |
| --- | --- |
| `Breeding Place Detection.zip` (from `Z:\vistext-v3`) | Yes |
| `best.pt` (current model) | Yes |
| `negatives/` folder with your own photos of places with **no** breeding containers (clean yards, covered drums, dry gutters, streets, rooms) | Recommended |
| `hard_negatives/` folder with clean photos the model wrongly flagged (from `negatives_review/` of a previous run) | Recommended |

Then choose **Runtime > Change runtime type > T4 GPU** and run the cells in order. Settings are in the next cell.

In [ ]:
# ---- Settings ----
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/ARID-YOLO')
DATASET_ZIP = DRIVE_ROOT / 'Breeding Place Detection.zip'
BASELINE_PT = DRIVE_ROOT / 'best.pt'
OWN_NEGATIVES_DIR = DRIVE_ROOT / 'negatives'      # your own clean-scene photos (optional)
REVIEW_DIR = DRIVE_ROOT / 'negatives_review'      # images the current model flagged; check them by hand
HARD_NEGATIVES_DIR = DRIVE_ROOT / 'hard_negatives' # flagged images you confirmed are clean (most useful)
OUTPUT_DIR = DRIVE_ROOT / 'v2'

WORK = Path('/content/arid-v2')
DATA = WORK / 'data'                              # merged dataset
YOLO_DIR = Path('/content/yolov5')
RUN_NAME = 'arid_yolov5s_v2'

CLASSES = ['Bottle', 'Coconut-Exocarp', 'Drain-Inlet', 'Tire', 'Vase']  # order must not change

# Clean scenes from Places365 (validation split, 100 images per category, 256 px).
USE_PLACES365 = True
PLACES_PER_CATEGORY = 30
PLACES_CATEGORIES = [
    'alley', 'street', 'backyard', 'yard', 'patio', 'driveway', 'parking_lot',
    'courtyard', 'lawn', 'playground', 'porch', 'residential_neighborhood',
    'village', 'field_road', 'park', 'market/outdoor', 'kitchen', 'living_room',
    'bedroom', 'basketball_court/outdoor',
]
NEGATIVE_TEST_FRACTION = 0.2      # held out to measure false alarms
SCREEN_CONFIDENCE = 0.25          # same threshold the dashboard uses

# Extra Roboflow datasets (optional). Leave empty to skip.
# Find workspace/project/version in the dataset's Roboflow URL, e.g.
# universe.roboflow.com/luis-augusto-silva-bq4bv/mosquito-suh0p/dataset/<version>
ROBOFLOW_API_KEY = ''
EXTRA_ROBOFLOW = [
    # dict(workspace='luis-augusto-silva-bq4bv', project='mosquito-suh0p', version=1),
]
# Their class names (lowercase) -> A.R.I.D. class. Anything not listed is dropped.
CLASS_MAP = {
    'bottle': 'Bottle', 'bottles': 'Bottle', 'plastic bottle': 'Bottle',
    'coconut': 'Coconut-Exocarp', 'coconut-exocarp': 'Coconut-Exocarp', 'coconut shell': 'Coconut-Exocarp',
    'drain': 'Drain-Inlet', 'drain-inlet': 'Drain-Inlet', 'drain inlet': 'Drain-Inlet', 'storm drain': 'Drain-Inlet',
    'tire': 'Tire', 'tyre': 'Tire', 'tires': 'Tire', 'tire_with_water': 'Tire',
    'vase': 'Vase', 'flower pot': 'Vase', 'pot': 'Vase', 'vase_with_water': 'Vase',
}

# Training
EPOCHS = 150
PATIENCE = 30        # stop early if validation stops improving
BATCH = 16
SEED = 0

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
assert torch.cuda.is_available(), 'GPU is not enabled. Select Runtime > Change runtime type > T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))

for required in (DATASET_ZIP, BASELINE_PT):
    assert required.exists(), f'Upload this to Google Drive first: {required}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Inputs found.')

In [ ]:
%cd /content
!rm -rf yolov5
!git clone --depth 1 https://github.com/ultralytics/yolov5.git
%cd /content/yolov5
!pip install -qr requirements.txt onnx onnxruntime roboflow
import os
os.environ['WANDB_DISABLED'] = 'true'

In [ ]:
# Extract the original dataset and convert polygon labels to boxes (same as v1).
import random
import shutil
import zipfile

random.seed(SEED)


def to_box_lines(label_path, id_map=None):
    """Return YOLO box lines for a label file. Polygons become enclosing boxes.

    id_map maps source class ids to A.R.I.D. ids; ids missing from it are dropped.
    """
    lines = []
    for line in label_path.read_text().splitlines():
        parts = line.split()
        if not parts:
            continue
        class_id = int(parts[0])
        if id_map is not None:
            if class_id not in id_map:
                continue
            class_id = id_map[class_id]
        values = [float(v) for v in parts[1:]]
        if len(values) == 4:
            xc, yc, w, h = values
        elif len(values) >= 6 and len(values) % 2 == 0:
            xs, ys = values[0::2], values[1::2]
            xc, yc = (min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2
            w, h = max(xs) - min(xs), max(ys) - min(ys)
        else:
            raise ValueError(f'Unsupported annotation in {label_path}: {line}')
        lines.append(f'{class_id} {xc:.8f} {yc:.8f} {w:.8f} {h:.8f}')
    return lines


def add_image(image_path, lines, split, prefix='', name=None):
    """Copy one image and write its label file into the merged dataset."""
    name = name or prefix + image_path.name
    (DATA / split / 'images').mkdir(parents=True, exist_ok=True)
    (DATA / split / 'labels').mkdir(parents=True, exist_ok=True)
    shutil.copy2(image_path, DATA / split / 'images' / name)
    label = DATA / split / 'labels' / (Path(name).stem + '.txt')
    label.write_text('\n'.join(lines) + ('\n' if lines else ''))


if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True)
with zipfile.ZipFile(DATASET_ZIP) as archive:
    archive.extractall(WORK / 'base')
BASE = WORK / 'base' / 'Breeding Place Detection'

IMAGE_TYPES = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
for split in ('train', 'valid', 'test'):
    count = 0
    for image in sorted((BASE / split / 'images').iterdir()):
        if image.suffix.lower() not in IMAGE_TYPES:
            continue
        label = BASE / split / 'labels' / (image.stem + '.txt')
        add_image(image, to_box_lines(label) if label.exists() else [], split)
        count += 1
    print(f'{split}: {count} original images')

In [ ]:
# Optional: merge extra Roboflow datasets. Their test images go to valid, so the
# original test split stays the fair benchmark.
import yaml

if EXTRA_ROBOFLOW:
    assert ROBOFLOW_API_KEY, 'Set ROBOFLOW_API_KEY in the settings cell.'
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)

for index, spec in enumerate(EXTRA_ROBOFLOW):
    location = WORK / f'extra{index}'
    rf.workspace(spec['workspace']).project(spec['project']).version(spec['version']).download(
        'yolov5', location=str(location))
    names = yaml.safe_load((location / 'data.yaml').read_text())['names']
    if isinstance(names, dict):
        names = [names[k] for k in sorted(names)]
    id_map = {i: CLASSES.index(CLASS_MAP[n.lower()]) for i, n in enumerate(names) if n.lower() in CLASS_MAP}
    print(f"{spec['project']}: classes {names}")
    print('  kept:', {names[i]: CLASSES[j] for i, j in id_map.items()})
    added = skipped = 0
    for src_split, dst_split in (('train', 'train'), ('valid', 'valid'), ('test', 'valid')):
        image_dir = location / src_split / 'images'
        if not image_dir.exists():
            continue
        for image in sorted(image_dir.iterdir()):
            label = location / src_split / 'labels' / (image.stem + '.txt')
            if not label.exists():
                continue
            lines = to_box_lines(label, id_map)
            if not lines:
                # Every object was a dropped class (e.g. bucket), so it is not a clean negative.
                skipped += 1
                continue
            add_image(image, lines, dst_split, prefix=f'x{index}_')
            added += 1
    print(f'  added {added} images, skipped {skipped} with only unmapped classes')

In [ ]:
# Collect clean-scene candidates: your own photos plus a Places365 sample.
candidates = []
if OWN_NEGATIVES_DIR.exists():
    own = [p for p in OWN_NEGATIVES_DIR.rglob('*') if p.suffix.lower() in IMAGE_TYPES]
    candidates += [('own', p) for p in own]
    print(f'Own negative photos: {len(own)}')
else:
    print(f'No {OWN_NEGATIVES_DIR} folder; using Places365 only.')

if USE_PLACES365:
    from torchvision.datasets import Places365
    places = Places365(str(WORK / 'places365'), split='val', small=True, download=True)
    wanted = {name: i for i, name in enumerate(places.classes) if name[3:] in PLACES_CATEGORIES}
    missing = set(PLACES_CATEGORIES) - {n[3:] for n in wanted}
    if missing:
        print('Categories not found in Places365:', sorted(missing))
    by_class = {}
    for path, class_index in places.imgs:
        by_class.setdefault(class_index, []).append(Path(path))
    for name, class_index in wanted.items():
        picked = random.sample(by_class[class_index], min(PLACES_PER_CATEGORY, len(by_class[class_index])))
        candidates += [('places', p) for p in picked]
    print(f'Places365 candidates: {sum(1 for s, _ in candidates if s == "places")}')

In [ ]:
# Screen candidates with the current model. An image where it detects something
# may really contain a bottle or pot, so it goes to a review folder instead of training.
import sys
sys.path.insert(0, str(YOLO_DIR))
%cd /content/yolov5


def load_detector(weights):
    model = torch.hub.load(str(YOLO_DIR), 'custom', path=str(weights), source='local', verbose=False)
    model.conf = SCREEN_CONFIDENCE
    return model


def images_with_detections(model, paths, batch=32):
    """Return the set of paths where the model finds at least one object."""
    flagged = set()
    for start in range(0, len(paths), batch):
        chunk = paths[start:start + batch]
        results = model([str(p) for p in chunk], size=640)
        for path, boxes in zip(chunk, results.xyxy):
            if len(boxes):
                flagged.add(path)
    return flagged


baseline = load_detector(BASELINE_PT)

# Hold out the false-alarm test set BEFORE screening, so it is not biased toward the
# current model (v2's first run held out only images the current model already passed).
random.shuffle(candidates)
held_out = int(len(candidates) * NEGATIVE_TEST_FRACTION)
test_pool, train_pool = candidates[:held_out], candidates[held_out:]
NEG_TEST = WORK / 'neg_test'
NEG_TEST.mkdir(parents=True, exist_ok=True)
for i, (source, path) in enumerate(test_pool):
    shutil.copy2(path, NEG_TEST / f'neg_{source}_{i:05d}{path.suffix.lower()}')

# Screen only the training pool. Flagged images may contain a real container, so they
# go to review; after you check them, the clean ones go into hard_negatives/.
flagged = images_with_detections(baseline, [p for _, p in train_pool])
if REVIEW_DIR.exists():
    shutil.rmtree(REVIEW_DIR)
REVIEW_DIR.mkdir(parents=True)
added = 0
for i, (source, path) in enumerate(train_pool):
    if path in flagged:
        shutil.copy2(path, REVIEW_DIR / f'{source}_{path.parent.name}_{path.name}')
    else:
        add_image(path, [], 'train', name=f'neg_{source}_{i:05d}{path.suffix.lower()}')  # empty label = background
        added += 1

# Hard negatives: images you checked by hand that contain NO breeding container, but that
# the model wrongly flagged. These teach it the most, so they are added without screening.
hard = [p for p in HARD_NEGATIVES_DIR.rglob('*') if p.suffix.lower() in IMAGE_TYPES] if HARD_NEGATIVES_DIR.exists() else []
for i, path in enumerate(hard):
    add_image(path, [], 'train', name=f'hard_{i:05d}{path.suffix.lower()}')

print(f'False-alarm test set (unscreened): {len(test_pool)} images')
print(f'Background images added to train: {added} screened + {len(hard)} hard negatives')
print(f'{len(flagged)} flagged images copied to {REVIEW_DIR}. Before the next run, move the ones with NO '
      f'breeding container into {HARD_NEGATIVES_DIR}; label real containers in Roboflow instead.')

In [ ]:
# Write data.yaml and summarize the merged dataset.
from collections import Counter

(DATA / 'data.yaml').write_text(yaml.safe_dump({
    'path': str(DATA), 'train': 'train/images', 'val': 'valid/images', 'test': 'test/images',
    'nc': len(CLASSES), 'names': CLASSES,
}, sort_keys=False))

for split in ('train', 'valid', 'test'):
    boxes, empty, total = Counter(), 0, 0
    for label in (DATA / split / 'labels').glob('*.txt'):
        total += 1
        rows = [l for l in label.read_text().splitlines() if l.strip()]
        if not rows:
            empty += 1
        boxes.update(CLASSES[int(r.split()[0])] for r in rows)
    print(f'{split}: {total} images, {empty} background ({empty / total:.0%}), boxes {dict(boxes)}')

In [ ]:
# Baseline scores: current model on the original test split and on held-out clean scenes.
import val as yolo_val

NEG_TEST_PATHS = sorted(NEG_TEST.iterdir())


def evaluate(weights, label):
    (mp, mr, map50, map5095, *_), per_class, _ = yolo_val.run(
        data=str(DATA / 'data.yaml'), weights=str(weights), imgsz=640, task='test',
        project=str(WORK / 'eval'), name=label, exist_ok=True, verbose=True, plots=True)
    detector = load_detector(weights)
    false_alarms = len(images_with_detections(detector, NEG_TEST_PATHS)) if NEG_TEST_PATHS else 0
    return {
        'precision': mp, 'recall': mr, 'mAP@50': map50, 'mAP@50-95': map5095,
        'clean scenes with a false alarm': f'{false_alarms}/{len(NEG_TEST_PATHS)}',
        'per-class mAP@50-95': dict(zip(CLASSES, per_class)),
    }


scores = {'v1 (current)': evaluate(BASELINE_PT, 'v1')}
scores['v1 (current)']

In [ ]:
# Fine-tune from the current model. Runs are written to the Colab disk, then copied to Drive.
import subprocess

subprocess.run([
    'python', 'train.py',
    '--img', '640', '--batch', str(BATCH), '--epochs', str(EPOCHS), '--patience', str(PATIENCE),
    '--data', str(DATA / 'data.yaml'),
    '--weights', str(BASELINE_PT),
    '--hyp', 'data/hyps/hyp.scratch-low.yaml',   # same augmentation as v1; scratch-med made the model over-detect
    '--project', str(WORK / 'runs'), '--name', RUN_NAME, '--exist-ok',
    '--workers', '2', '--seed', str(SEED),
], cwd=YOLO_DIR, check=True)

RUN_DIR = WORK / 'runs' / RUN_NAME
shutil.copytree(RUN_DIR, OUTPUT_DIR / RUN_NAME, dirs_exist_ok=True)
NEW_PT = RUN_DIR / 'weights' / 'best.pt'
print('Saved to Drive:', OUTPUT_DIR / RUN_NAME)

In [ ]:
# Compare v1 and v2 on the same test split and the same clean scenes.
import pandas as pd

scores['v2 (new)'] = evaluate(NEW_PT, 'v2')
table = pd.DataFrame({k: {m: v for m, v in s.items() if m != 'per-class mAP@50-95'} for k, s in scores.items()})
per_class = pd.DataFrame({k: s['per-class mAP@50-95'] for k, s in scores.items()})
display(table)
display(per_class.rename_axis('mAP@50-95'))
table.to_csv(OUTPUT_DIR / 'comparison.csv')
per_class.to_csv(OUTPUT_DIR / 'comparison_per_class.csv')
print('Use v2 only if mAP@50 on the test split is about the same or higher AND it has fewer false alarms.')

In [ ]:
# Export for the dashboard (same settings as v1) and check the output shape.
import onnxruntime as ort
import numpy as np

subprocess.run(['python', 'export.py', '--weights', str(NEW_PT), '--include', 'onnx',
                '--img', '640', '--opset', '12'], cwd=YOLO_DIR, check=True)
onnx_path = NEW_PT.with_suffix('.onnx')
session = ort.InferenceSession(str(onnx_path))
output = session.run(None, {session.get_inputs()[0].name: np.zeros((1, 3, 640, 640), np.float32)})[0]
assert output.shape == (1, 25200, 10), f'Unexpected output shape {output.shape}; the dashboard expects (1, 25200, 10)'

final_onnx = OUTPUT_DIR / 'medsam_yolov5s.onnx'
shutil.copy2(onnx_path, final_onnx)
shutil.copy2(NEW_PT, OUTPUT_DIR / 'best.pt')
print('Ready for the dashboard:', final_onnx, output.shape)

from google.colab import files
files.download(str(final_onnx))